In [13]:
import os
from Bio import SeqIO
from Bio.Seq import Seq

def translate_all_frames(dna_seq, min_protein_length=30):
    """Traduce la secuencia en los 3 marcos de lectura y devuelve la proteína más larga"""
    results = {}
    for frame in range(3):
        # Traducción normal (hasta el primer codón STOP)
        protein = dna_seq[frame:].translate(to_stop=True)
        
        # Traducción completa (mostrando STOPs) para análisis
        full_protein = dna_seq[frame:].translate(to_stop=False)
        stop_positions = [i for i, aa in enumerate(full_protein) if aa == '*']
        
        results[frame] = {
            'protein': protein,
            'length': len(protein),
            'full_translation': full_protein,
            'stop_positions': stop_positions
        }
    
    # Seleccionar el marco con la proteína más larga
    best_frame = max(results.items(), key=lambda x: x[1]['length'])
    
    # Si la proteína más larga es muy corta, devolver todas las opciones
    if best_frame[1]['length'] < min_protein_length:
        return {'status': 'warning', 'message': 'All translations are very short', 'all_frames': results}
    else:
        return {'status': 'success', 'best_frame': best_frame, 'all_frames': results}

def process_sequence_file(input_path, output_path):
    """Procesa un archivo FASTA individual"""
    with open(input_path, "r") as input_file, open(output_path, "w") as output_file:
        for record in SeqIO.parse(input_file, "fasta"):
            dna_sequence = str(record.seq).upper().replace(" ", "").replace("\n", "")
            dna_seq = Seq(dna_sequence)
            
            # Análisis de traducción
            translation_result = translate_all_frames(dna_seq)
            
            # Escribir resultados
            output_file.write(f">{record.id}_translation_report\n")
            
            if translation_result['status'] == 'success':
                best = translation_result['best_frame']
                output_file.write(f"Best frame: {best[0]}\n")
                output_file.write(f"Protein length: {best[1]['length']} aa\n")
                output_file.write(f"Protein sequence:\n{str(best[1]['protein'])}\n\n")
                
                # Escribir información sobre otros marcos
                output_file.write("All frames analysis:\n")
                for frame, data in translation_result['all_frames'].items():
                    output_file.write(f"Frame {frame}: {data['length']} aa | ")
                    output_file.write(f"STOP positions: {data['stop_positions']}\n")
            else:
                output_file.write("WARNING: All translations are very short (<30 aa)\n")
                output_file.write("Showing all frames:\n")
                for frame, data in translation_result['all_frames'].items():
                    output_file.write(f"Frame {frame}: {data['protein']} (len={data['length']})\n")
                    output_file.write(f"Full translation: {data['full_translation']}\n")
                    output_file.write(f"STOP positions: {data['stop_positions']}\n\n")

def translate_dna_to_protein(input_folder, output_folder):
    """Procesa todos los archivos en la carpeta de entrada"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for filename in os.listdir(input_folder):
        if filename.endswith(".fasta") and filename.startswith("sequence_"):
            input_path = os.path.join(input_folder, filename)
            output_filename = f"protein_analysis_{filename.split('_')[1]}"
            output_path = os.path.join(output_folder, output_filename)
            
            print(f"\nProcessing {filename}...")
            process_sequence_file(input_path, output_path)
            print(f"Analysis saved to {output_filename}")

if __name__ == "__main__":
    # Configuración de rutas
    input_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences"
    output_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\secuencias_a_proteinas"
    
    print("=== DNA to Protein Translation Tool ===")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}\n")
    
    translate_dna_to_protein(input_folder, output_folder)
    print("\nTranslation complete!")

=== DNA to Protein Translation Tool ===
Input folder: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences
Output folder: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\secuencias_a_proteinas


Processing sequence_1.fasta...
Analysis saved to protein_analysis_1.fasta

Processing sequence_10.fasta...
Analysis saved to protein_analysis_10.fasta

Processing sequence_11.fasta...
Analysis saved to protein_analysis_11.fasta

Processing sequence_12.fasta...
Analysis saved to protein_analysis_12.fasta

Processing sequence_13.fasta...
Analysis saved to protein_analysis_13.fasta

Processing sequence_2.fasta...
Analysis saved to protein_analysis_2.fasta

Processing sequence_3.fasta...
Analysis saved to protein_analysis_3.fasta

Proce

In [14]:
import os
from Bio import SeqIO
from Bio.Seq import Seq

def translate_all_frames(dna_seq, record_id):
    """Traduce la secuencia en los 3 marcos de lectura"""
    results = {}
    for frame in range(3):
        # Traducción completa (incluyendo STOPs)
        protein_seq = dna_seq[frame:].translate(to_stop=False)
        
        # Guardar resultados
        results[frame] = {
            'sequence': protein_seq,
            'stops': [i+1 for i, aa in enumerate(protein_seq) if aa == '*'],  # Posiciones 1-based
            'length': len(protein_seq)
        }
    return results

def process_sequence_file(input_path, output_folder, seq_id):
    """Procesa un archivo FASTA y guarda traducciones por marco"""
    with open(input_path, "r") as input_file:
        for record in SeqIO.parse(input_file, "fasta"):
            dna_sequence = str(record.seq).upper().replace(" ", "").replace("\n", "")
            dna_seq = Seq(dna_sequence)
            
            # Traducir en los 3 marcos
            translations = translate_all_frames(dna_seq, record.id)
            
            # Crear archivos separados para cada marco
            for frame, data in translations.items():
                output_filename = f"{seq_id}_frame_{frame+1}.fasta"  # Marcos 1-3 en lugar de 0-2
                output_path = os.path.join(output_folder, output_filename)
                
                with open(output_path, "w") as output_file:
                    output_file.write(f">{record.id}_frame_{frame+1}\n")
                    # Dividir la secuencia en líneas de 80 caracteres
                    protein_seq = str(data['sequence'])
                    for i in range(0, len(protein_seq), 80):
                        output_file.write(protein_seq[i:i+80] + "\n")

def translate_dna_to_protein(input_folder, output_folder):
    """Procesa todos los archivos en la carpeta de entrada"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for filename in sorted(os.listdir(input_folder)):
        if filename.endswith(".fasta") and filename.startswith("sequence_"):
            seq_id = filename.split('_')[1].split('.')[0]  # Extraer el número de secuencia
            input_path = os.path.join(input_folder, filename)
            
            print(f"\nProcessing {filename}...")
            process_sequence_file(input_path, output_folder, seq_id)
            print(f"Generated 3 frame translations for sequence {seq_id}")

if __name__ == "__main__":
    # Configuración de rutas
    input_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences"
    output_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\secuencias_por_frame"
    
    print("=== DNA to Protein Translation by Frame ===")
    print(f"Input folder: {input_folder}")
    print(f"Output folder: {output_folder}\n")
    
    translate_dna_to_protein(input_folder, output_folder)
    print("\nTranslation complete! All frames generated for each sequence.")

=== DNA to Protein Translation by Frame ===
Input folder: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\data\Identificacion_patogeno\identifiacion_patogeno_individual_sequences
Output folder: C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\secuencias_por_frame


Processing sequence_1.fasta...
Generated 3 frame translations for sequence 1

Processing sequence_10.fasta...
Generated 3 frame translations for sequence 10

Processing sequence_11.fasta...
Generated 3 frame translations for sequence 11

Processing sequence_12.fasta...
Generated 3 frame translations for sequence 12

Processing sequence_13.fasta...
Generated 3 frame translations for sequence 13

Processing sequence_2.fasta...
Generated 3 frame translations for sequence 2

Processing sequence_3.fasta...
Generated 3 frame translation

In [15]:
import os
from Bio import SeqIO

def combine_frames_with_query(sequences_folder, queries_folder, output_folder):
    """
    Combina los frames de cada secuencia con su query correspondiente
    
    Args:
        sequences_folder: Carpeta con los archivos _frame_[1-3].fasta
        queries_folder: Carpeta con los archivos Query_[1-13].fasta
        output_folder: Carpeta para los archivos combinados
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Procesar cada secuencia del 1 al 13
    for seq_num in range(1, 14):
        # Archivos a combinar
        files_to_combine = []
        
        # Añadir los 3 frames
        for frame in [1, 2, 3]:
            frame_file = f"{seq_num}_frame_{frame}.fasta"
            frame_path = os.path.join(sequences_folder, frame_file)
            if os.path.exists(frame_path):
                files_to_combine.append(frame_path)
        
        # Añadir el query correspondiente
        query_file = f"Query_{seq_num}.fasta"
        query_path = os.path.join(queries_folder, query_file)
        if os.path.exists(query_path):
            files_to_combine.append(query_path)
        else:
            print(f"Advertencia: No se encontró {query_file}")
        
        # Solo proceder si hay archivos para combinar
        if files_to_combine:
            # Archivo de salida
            output_file = f"Combined_{seq_num}.fasta"
            output_path = os.path.join(output_folder, output_file)
            
            # Combinar los archivos
            with open(output_path, "w") as out_handle:
                for file in files_to_combine:
                    with open(file, "r") as in_handle:
                        for line in in_handle:
                            out_handle.write(line)
            
            print(f"Archivo combinado creado: {output_file}")
        else:
            print(f"No se encontraron archivos para la secuencia {seq_num}")

if __name__ == "__main__":
    # Configurar rutas (ajustar según tu estructura)
    frames_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\secuencias_por_frame"  # Carpeta con los _frame_[1-3].fasta
    queries_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\Hits_a_fasta"  # Carpeta con los Query_[1-13].fasta
    output_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\Combinados_query_hits"  # Carpeta para los archivos combinados
    
    print("=== Combinando frames con queries ===")
    combine_frames_with_query(frames_folder, queries_folder, output_folder)
    print("\nProceso completado!")

=== Combinando frames con queries ===
Advertencia: No se encontró Query_1.fasta
Archivo combinado creado: Combined_1.fasta
Advertencia: No se encontró Query_2.fasta
Archivo combinado creado: Combined_2.fasta
Archivo combinado creado: Combined_3.fasta
Archivo combinado creado: Combined_4.fasta
Archivo combinado creado: Combined_5.fasta
Archivo combinado creado: Combined_6.fasta
Archivo combinado creado: Combined_7.fasta
Archivo combinado creado: Combined_8.fasta
Archivo combinado creado: Combined_9.fasta
Archivo combinado creado: Combined_10.fasta
Archivo combinado creado: Combined_11.fasta
Archivo combinado creado: Combined_12.fasta
Archivo combinado creado: Combined_13.fasta

Proceso completado!


In [1]:
import os
from Bio.Align.Applications import ClustalOmegaCommandline

def run_clustal_aln_format(input_file, output_file):
    """
    Ejecuta Clustal Omega y genera la salida en formato .aln (CLUSTAL)
    """
    clustalomega_cline = ClustalOmegaCommandline(
        infile=input_file,
        outfile=output_file,
        verbose=True,
        auto=True,
        seqtype="protein",
        outfmt="clu"
    )
    clustalomega_cline()

def process_all_alignments(input_folder, output_folder):
    """Procesa todos los archivos combinados del 1 al 13"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for i in range(1, 14):
        input_file = os.path.join(input_folder, f"Combined_{i}.fasta")
        output_file = os.path.join(output_folder, f"Aligned_{i}.aln")
        
        if os.path.exists(input_file):
            print(f"Procesando secuencia {i}...")
            run_clustal_aln_format(input_file, output_file)
            print(f"Resultado guardado en Aligned_{i}.aln")
        else:
            print(f"Archivo no encontrado: Combined_{i}.fasta")

if __name__ == "__main__":
    input_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\Combinados_query_hits"
    output_folder = r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\results\2025-03-30_FG\identificacion_patogeno\identificacion_patogeno_4_02.07.25\Alineamiento_multiple_3.0_traducido\alineamiento_multiple"
    
    print("=== Generando alineamientos en formato .aln ===")
    process_all_alignments(input_folder, output_folder)
    print("\nProceso completado!")


c:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\.venv\Lib\site-packages\Bio\Application\__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


=== Generando alineamientos en formato .aln ===
Procesando secuencia 1...
Resultado guardado en Aligned_1.aln
Procesando secuencia 2...
Resultado guardado en Aligned_2.aln
Procesando secuencia 3...
Resultado guardado en Aligned_3.aln
Procesando secuencia 4...
Resultado guardado en Aligned_4.aln
Procesando secuencia 5...
Resultado guardado en Aligned_5.aln
Procesando secuencia 6...
Resultado guardado en Aligned_6.aln
Procesando secuencia 7...
Resultado guardado en Aligned_7.aln
Procesando secuencia 8...
Resultado guardado en Aligned_8.aln
Procesando secuencia 9...
Resultado guardado en Aligned_9.aln
Procesando secuencia 10...
Resultado guardado en Aligned_10.aln
Procesando secuencia 11...
Resultado guardado en Aligned_11.aln
Procesando secuencia 12...
Resultado guardado en Aligned_12.aln
Procesando secuencia 13...
Resultado guardado en Aligned_13.aln

Proceso completado!
